# Repairing an inconsistent SpatialData Zarr store

This notebook reproduces a broken `SpatialData` store where the image data on disk is removed, but the root `zarr.json` still advertises that image. It then shows how that inconsistency affects `read_zarr()` and how `delete_element_from_disk()` brings the store metadata back in sync.

In [43]:
import harpy as hp
from spatialdata import read_zarr

sdata = hp.datasets.resolve_example()

sdata.write("/Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/", overwrite=True)

sdata = read_zarr(sdata.path)
sdata

/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy/lib/python3.12/site-packages/harpy/datasets/transcriptomics.py:36: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = read_zarr(os.path.commonpath(unzip_path))
no parent found for <ome_zarr.reader.Label object at 0x16acbf9e0>: None
no parent found for <ome_zarr.reader.Label object at 0x313543770>: None
no parent found for <ome_zarr.reader.Label object at 0x16aa533b0>: None
no parent found for <ome_zarr.reader.Label object at 0x16aa78c80>: None


SpatialData object, with associated Zarr store: /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr
├── Images
│     └── 'raw_image': DataArray[cyx] (1, 4288, 2144)
├── Labels
│     ├── 'segmentation_mask': DataArray[yx] (4288, 2144)
│     └── 'segmentation_mask_expanded': DataArray[yx] (4288, 2144)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'filtered_low_counts_segmentation_mask_boundaries': GeoDataFrame shape: (33, 1) (2D shapes)
│     ├── 'filtered_segmentation_segmentation_mask_boundaries': GeoDataFrame shape: (8, 1) (2D shapes)
│     └── 'segmentation_mask_boundaries': GeoDataFrame shape: (616, 1) (2D shapes)
└── Tables
      ├── 'table_transcriptomics': AnnData (649, 96)
      ├── 'table_transcriptomics_cluster': AnnData (616, 87)
      └── 'table_transcriptomics_preprocessed': AnnData (616, 87)
with coordinate systems:
    ▸ 'global', with elements:
        raw_image (Images), segmentation_mask (Labels), segmentation_ma

## Break the store on purpose

Here we define a small helper that checks whether an element is still referenced in the root `zarr.json`. After that, we manually delete the `images/raw_image` directory to create the mismatch: the files are gone, but the metadata entry is still present.

In [44]:
import json
import shutil
from pathlib import Path


def has_element_in_zarr_json(zarr_json_path: str | Path, element_name: str) -> bool:
    zarr_json_path = Path(zarr_json_path)

    with zarr_json_path.open() as f:
        zarr_json = json.load(f)

    metadata = zarr_json.get("consolidated_metadata", {}).get("metadata", {})
    return f"images/{element_name}" in metadata or any(
        key.endswith(f"/{element_name}") for key in metadata
    )


zarr_json_path = Path("/Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/zarr.json")


# lets break the spatialdata object:
shutil.rmtree(
    "/Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/images/raw_image"
)  # now the zarr.json is not in sync with what is actual in the zarr store

print(
    f"raw_image present in {zarr_json_path}: {has_element_in_zarr_json(zarr_json_path, 'raw_image')}"
)  # -> raw image still present in our zarr.json

raw_image present in /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/zarr.json: True


## Observe the broken behavior

With `on_bad_files="error"`, `read_zarr()` fails immediately because the metadata still points to `raw_image` even though its directory no longer exists. With `on_bad_files="warn"`, the store can still be opened, but the warning confirms the store is inconsistent.

In [45]:
from spatialdata import read_zarr

sdata = read_zarr(sdata.path, on_bad_files="error")

OSError: Image location /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/images/raw_image does not seem to exist. If it does, potentially the zarr.json (or .zattrs) file inside is corrupted or not present or the image files themselves are corrupted.

In [46]:
sdata = read_zarr(sdata.path, on_bad_files="warn")

/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy/lib/python3.12/site-packages/spatialdata/_io/io_raster.py:51: UserWarning: images/raw_image: OSError: Image location /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/images/raw_image does not seem to exist. If it does, potentially the zarr.json (or .zattrs) file inside is corrupted or not present or the image files themselves are corrupted.
  raise OSError(
no parent found for <ome_zarr.reader.Label object at 0x16aa1d490>: None
no parent found for <ome_zarr.reader.Label object at 0x17fe0f1d0>: None


## Repair the metadata

Calling `delete_element_from_disk("raw_image")` removes the stale element registration from the store metadata. The next cell reuses the helper function to confirm that `raw_image` is no longer listed in the root `zarr.json`.

In [47]:
sdata.delete_element_from_disk("raw_image")  # this fixes the zarr.json

In [48]:
print(
    f"raw_image present in {zarr_json_path}: {has_element_in_zarr_json(zarr_json_path, 'raw_image')}"
)

raw_image present in /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr/zarr.json: False


## Verify the fix

Once the stale metadata entry is gone, `read_zarr(..., on_bad_files="error")` succeeds again. The final output shows the repaired `SpatialData` object without the removed `raw_image` element.

In [50]:
sdata = read_zarr(sdata.path, on_bad_files="error")

no parent found for <ome_zarr.reader.Label object at 0x3134e7680>: None
no parent found for <ome_zarr.reader.Label object at 0x16b071040>: None


In [ ]:
sdata  # -> all good!

SpatialData object, with associated Zarr store: /Users/arne.defauw/VIB/DATA/test_data/sdata.zarr
├── Labels
│     ├── 'segmentation_mask': DataArray[yx] (4288, 2144)
│     └── 'segmentation_mask_expanded': DataArray[yx] (4288, 2144)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'filtered_low_counts_segmentation_mask_boundaries': GeoDataFrame shape: (33, 1) (2D shapes)
│     ├── 'filtered_segmentation_segmentation_mask_boundaries': GeoDataFrame shape: (8, 1) (2D shapes)
│     └── 'segmentation_mask_boundaries': GeoDataFrame shape: (616, 1) (2D shapes)
└── Tables
      ├── 'table_transcriptomics': AnnData (649, 96)
      ├── 'table_transcriptomics_cluster': AnnData (616, 87)
      └── 'table_transcriptomics_preprocessed': AnnData (616, 87)
with coordinate systems:
    ▸ 'global', with elements:
        segmentation_mask (Labels), segmentation_mask_expanded (Labels), transcripts (Points), filtered_low_counts_segmentation_mask_bou